In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import math

In [ ]:
train_data = pd.read_csv('data/train.csv')

In [ ]:
test_data = pd.read_csv('data/test.csv')

In [4]:
import re
import math
from collections import Counter

_ARABIC_TO_PERSIAN_MAP = str.maketrans({
    "ي": "ی",
    "ك": "ک",
    "ۀ": "ه",
    "ة": "ه",
    "ؤ": "و",
    "إ": "ا",
    "أ": "ا",
    "ٱ": "ا",
})

_DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")
_URL_EMAIL_RE = re.compile(r"(https?://\S+|www\.\S+|\S+@\S+)")
_DIGITS_RE = re.compile(r"[0-9\u06F0-\u06F9\u0660-\u0669]+")
_NON_LETTER_RE = re.compile(r"[^\u0600-\u06FF\s\u200c]+")

_STOPWORDS = {
    "و","در","به","از","با","که","این","آن","برای","را","تا","اما","یا","نیز","هم","خود",
    "من","تو","او","ما","شما","ایشان","اینا","اونا","ماها","شماها",
    "یک","یه","همه","هر","هیچ","چند","چون","اگر","پس","قبل","بعد","روی","مثل","بین",
    "است","هست","بود","شد","می","می‌شود","می‌شه","میباشد","باشد","باشه","نیست","ندارد","دارم","دارید","دارن",
    "کرد","کردم","کردی","کرده","کردن","کنم","کنی","کند","کنید","کنن",
    "خیلی","بسیار","واقعا","تقریبا","کمی","کم","زیاد","زیادی",
    "فقط","حتی","البته","اصلا","اصلاً","کلا",
}

_SUFFIXES = ("هایی", "های", "ها")

def preprocessing(text):
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return []
    text = str(text)

    text = _URL_EMAIL_RE.sub(" ", text)
    text = text.translate(_ARABIC_TO_PERSIAN_MAP)
    text = _DIACRITICS_RE.sub("", text).replace("ـ", "")
    text = text.replace("\u200c", " ")
    text = _DIGITS_RE.sub(" NUMBER ", text)
    text = _NON_LETTER_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []
    for tok in text.split():
        if not tok or tok in _STOPWORDS:
            continue
        for suf in _SUFFIXES:
            if tok.endswith(suf) and len(tok) > len(suf) + 1:
                tok = tok[:-len(suf)]
                break
        if tok != "NUMBER" and len(tok) < 2:
            continue
        tokens.append(tok)
    return tokens

In [5]:
label_counts = train_data['price_value'].value_counts()
total = len(train_data)
prior_probability = {
    0: label_counts.get(0, 0) / total,
    1: label_counts.get(1, 0) / total,
}

In [6]:
def token_counter(texts):
    count_dict = {}
    for t in texts:
        for tok in preprocessing(t):
            count_dict[tok] = count_dict.get(tok, 0) + 1
    return count_dict

In [7]:
negative_texts = train_data.loc[train_data['price_value'] == 0, 'comment'].values
negative_class_count = token_counter(negative_texts)

In [8]:
positive_texts = train_data.loc[train_data['price_value'] == 1, 'comment'].values
positive_class_count = token_counter(positive_texts)

vocab = set(negative_class_count.keys()) | set(positive_class_count.keys())
vocab_size = len(vocab)
neg_total_tokens = sum(negative_class_count.values())
pos_total_tokens = sum(positive_class_count.values())

In [ ]:
def compute_probability(text, cls):
    tokens = preprocessing(text)
    logp = math.log(prior_probability[cls] + 1e-12)

    if cls == 0:
        class_count = negative_class_count
        denom = neg_total_tokens + vocab_size
    else:
        class_count = positive_class_count
        denom = pos_total_tokens + vocab_size

    for tok in tokens:
        num = class_count.get(tok, 0) + 1
        logp += math.log(num / denom)
    return logp

In [10]:
def predict(test):
    predictions = []
    for text in test:
        p0 = compute_probability(text, 0)
        p1 = compute_probability(text, 1)
        predictions.append(1 if p1 > p0 else 0)
    return np.array(predictions)

In [11]:
train_predictions = predict(train_data['comment'].values)
accuracy_score(train_data['price_value'].values, train_predictions)

0.894175